# Experiment 4: analysis notebook

10060 Physics (Polytechnical Foundation), lab on Friday 23 October 2026.

This notebook reads the CSV files you export from the iOLab app and computes the moment of inertia of the iOLab about its $y$ axis in two independent ways:

* **Method N, rotational Newton's second law.** Up and down runs on the ramp. The slopes of $\omega(t)$ give $\alpha_{up}$ and $\alpha_{down}$. Their mean cancels rolling resistance, and $I_P=MgR\sin\theta/\bar\alpha$.
* **Method E, energy.** The body is released from rest at a gate and rolls a measured distance $s_{AB}$ to the end of the board. The angular velocity there, $\omega_B$, gives $I_P=2Mg\,s_{AB}\sin\theta\,(1-\varepsilon)/\omega_B^2$.

In both cases $I_P=I+MR^2$ is the moment of inertia about the contact line, $I$ is the whole body, and the iOLab alone is $I_{iOLab}=I-I_{rolls}$.

It also handles the gyroscope calibration roll, the ramp angle from the accelerometer, the tape rolls, all uncertainties (with a budget that shows which input matters), and the report figures. It writes no report text.

**How to use it**

1. Next to this notebook, make the folders `data/calibration/roll1`, `roll2`, ..., `data/angle`, `data/table`, `data/newton/run1`, `run2`, ..., and `data/energy/run1`, `run2`, ...
2. After every recording, export the chart you need (toolbar icon at the bottom left of the chart, then Export to CSV). The file lands in `Documents/iOLab-WorkFiles/export`. Move it into the folder of that run. Gyroscope files go into calibration, newton and energy. Accelerometer files go into angle and table.
3. Fill in the settings cell with your measured numbers, and set `DATA_DIR = Path("data")`.
4. Run everything (menu Run, then Run All Cells).
5. Look at the plot of every run. If a window or a corner looks wrong, change the setting and run again.

Out of the box it runs on `example_data`, a simulated data set made with a realistic model of the iOLab. Run it once before the lab to see what every step prints.

Packages: numpy, pandas, matplotlib. Install with `pip install numpy pandas matplotlib` if they are missing.

In [ ]:
import re
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("numpy", np.__version__, "| pandas", pd.__version__)

## Settings

The only cell you normally edit. Every number has its unit in the comment. The values below are the ones that belong to `example_data`. Replace all of them with your own.

In [ ]:
# ============================== SETTINGS: edit this cell only ==============================
DATA_DIR = Path("example_data")      # Path("data") in the lab. "example_data" is the simulated test set.
OUT_DIR = Path("output")             # figures and tables are written here

g = 9.82                             # m/s^2, the course value

# 1. The parts. Uncertainty = scale resolution or reading uncertainty.
m_iolab, dm_iolab = 0.1962, 0.0001   # kg, iOLab with batteries, without the USB dongle
m_roll = [0.1842, 0.1878]            # kg, tape roll 1 and tape roll 2
dm_roll = 0.0001                     # kg
M_body, dM_body = 0.5682, 0.0001     # kg, the whole body with both rolls on. None if you did not weigh it.
D_in = [0.0762, 0.0762]              # m, inner diameter of each roll (inside of the cardboard core)
dD_in = 0.0005                       # m
C_out = [0.3346, 0.3353]             # m, outer circumference of each roll (paper strip around the roll)
dC_out = 0.0010                      # m
iolab_w, iolab_t, d_box = 0.0752, 0.0301, 0.0005   # m, iOLab width (x) and thickness (z), for the box model
CORE_WALL = 0.0030                   # m, wall of the cardboard core, only for the two layer check. None to skip.

# 2. Calibration roll on the flat table: CAL_TURNS full turns, CAL_DIST = distance between the two table marks.
CAL_TURNS = 3
CAL_DIST = [1.006, 1.003, 1.003]     # m, one value per folder roll1, roll2, ... in the same order
dCAL_DIST = 0.002                    # m, ruler reading plus placing the pen line on the mark
ALIGN_MM = 1.0                       # mm, how well the pen line lands on the mark at start and end

# 3. Ramp angle. Accelerometer files in DATA_DIR/angle and DATA_DIR/table are used automatically.
RULER = dict(L=0.800, h_low=0.0291, h_high=0.1979, dL=0.001, dh=0.0005)   # m, the ruler check. None to skip.
ANGLE_SOURCE = "auto"                # "auto" (accelerometer if present, else ruler), "accelerometer" or "ruler"

# 4. Energy runs: distance ALONG the board from the gate face to the lower end of the board.
D_GATE, dD_GATE = 0.4470, 0.0010     # m. The contact point at release is R further up: s_AB = D_GATE + R.

# Which radius to use for R: "roll" (calibration roll, best) or "geometry" (circumference of the rolls)
R_SOURCE = "roll"

# 5. Air drag estimate (small correction). Length of the whole body along its axis, outside edge to outside edge.
BODY_LENGTH = 0.160                  # m
C_D, dC_D = 1.0, 0.5                 # drag coefficient of a short cylinder moving sideways: a rough estimate

# Analysis choices. The defaults are fine for most runs.
BASELINE_S = 0.5                     # s of still data at the start of every recording (hold still 1 s)
SMOOTH_MS = 10                       # ms, moving average used only to LOCATE points; fits use raw data
MARGIN_PUSH_S = 0.08                 # s skipped after the push has ended
MARGIN_TURN_S = 0.05                 # s skipped on both sides of the turning point
MARGIN_END_S = 0.05                  # s skipped before the hand catches the body
MANUAL_WINDOWS = {}                  # e.g. {"run3": {"up": (1.40, 1.85), "down": (2.00, 2.55)}}
MANUAL_CORNER = {}                   # energy runs, e.g. {"run2": 2.153} = time (s) the body leaves the board
GYRO_AXIS = "auto"                   # "auto" or "x", "y", "z". The rolling axis should be y.
GYRO_UNITS = "auto"                  # "auto", "rad/s" or "deg/s". auto: the calibration roll tells which.
REPORT_RUN_NEWTON = None             # e.g. "run2" for the report figure (None = the first run)
REPORT_RUN_ENERGY = None

# Manual mode: use these ONLY if you read the numbers in the iOLab app instead of exporting CSV files.
# Calibration roll, one dict per roll. area = "a" of a selection from the still part before to the still part after,
# offset = mu of a still selection, width = length of the area selection (s).
#   MANUAL_CAL = [dict(area=19.06, offset=-0.021, width=4.9, d=1.006)]
MANUAL_CAL = []
# Angle: mu of Ay held upright (y up, y down), then one (y up the slope, y down the slope) pair per place on the board.
#   MANUAL_ANGLE = dict(Ay_vertical_up=9.80, Ay_vertical_down=-9.70, pairs=[(2.155, -2.033), (2.149, -2.027)],
#                       sd=0.022, width=3.0, f=200)
MANUAL_ANGLE = {}
# Newton runs: slope s of a selection on the up window and on the down window of the gyroscope graph (rad/s^2).
#   MANUAL_NEWTON = [dict(s_up=-25.71, s_down=-24.98, w_max=17.2)]   (w_max = largest |omega| right after the push)
MANUAL_NEWTON = []
# Energy runs: a selection on the last 0.15 s of the rise that ends exactly at the corner: mu, s and its width.
#   MANUAL_ENERGY = [dict(mu=-23.55, s=-25.1, width=0.15)]      (omega_B = |mu| + |s| width / 2)
MANUAL_ENERGY = []

## Helpers

Reading the iOLab CSV files (a `time` column and `cal[0]`, `cal[1]`, `cal[2]` columns), rounding with the course rule (uncertainty with one significant figure, two if it starts with 1), the standard score
$$z=\frac{|x_1-x_2|}{\sqrt{\delta x_1^2+\delta x_2^2}}$$
with the course limits ($z\le1$ very likely the same, $1<z\le2$ likely the same, $2<z\le3$ grey zone, $z>3$ unlikely the same), straight line fits with the standard error of the slope, and error propagation for independent inputs,
$$\delta f=\sqrt{\sum_i\left(\frac{\partial f}{\partial x_i}\,\delta x_i\right)^2},$$
which also gives the uncertainty budget: the size of each term in the sum.

In [ ]:
# ============================== HELPERS: no need to edit ==============================
OUT_DIR.mkdir(parents=True, exist_ok=True)
BLUE, ORANGE, AQUA, INK, MUTED, GREY = "#2a78d6", "#eb6834", "#1baf7a", "#0b0b0b", "#8a8984", "#e4e3df"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": GREY, "grid.linewidth": 0.5, "font.size": 9})


def fmt(x, dx, unit=""):
    """Round the uncertainty to 1 significant figure (2 if it starts with 1) and the value to match."""
    if x is None or not np.isfinite(x):
        return "n/a"
    if dx is None or not np.isfinite(dx) or dx <= 0:
        return f"{x:.4g} {unit}".strip()
    e = int(math.floor(math.log10(dx)))
    nsig = 2 if dx / 10 ** e < 2 else 1
    dec = nsig - 1 - e
    if dec > 0:
        return f"{x:.{dec}f} ± {dx:.{dec}f} {unit}".strip()
    r = 10 ** (-dec)
    return f"{round(x / r) * r:.0f} ± {round(dx / r) * r:.0f} {unit}".strip()


def fmt_e(x, dx, unit="", exp=-4):
    """Same rounding, written as (a ± b) x 10^exp unit."""
    return f"({fmt(x / 10 ** exp, dx / 10 ** exp)}) × 10^{exp} {unit}".strip()


def plural(n, word):
    return f"{n} {word}" if n == 1 else f"{n} {word}s"


def zscore(x1, dx1, x2=0.0, dx2=0.0):
    den = math.sqrt(dx1 ** 2 + dx2 ** 2)
    return abs(x1 - x2) / den if den > 0 else float("inf")


def verdict(z):
    if z <= 1:
        return "very likely the same (z <= 1)"
    if z <= 2:
        return "likely the same (1 < z <= 2)"
    if z <= 3:
        return "grey zone (2 < z <= 3)"
    return "unlikely the same (z > 3)"


def mean_sd_sdom(values):
    v = np.asarray([x for x in values if x is not None and np.isfinite(x)], dtype=float)
    n = v.size
    if n == 0:
        return np.nan, np.nan, np.nan, 0
    sd = v.std(ddof=1) if n > 1 else np.nan
    return v.mean(), sd, (sd / math.sqrt(n) if n > 1 else np.nan), n


def mean_with_floor(values, single_uncs):
    """Mean over runs. Uncertainty = SDOM, but never smaller than the typical single run uncertainty / sqrt(n)."""
    mu, sd, sdom, n = mean_sd_sdom(values)
    su = [u for u in single_uncs if u is not None and np.isfinite(u)]
    floor = math.sqrt(np.mean(np.square(su))) / math.sqrt(n) if (n and su) else 0.0
    unc = floor if (n < 2 or not np.isfinite(sdom)) else max(sdom, floor)
    return mu, unc, n


def moving_average(y, n):
    y = np.asarray(y, dtype=float)
    if n <= 1:
        return y.copy()
    k = np.ones(n) / n
    ypad = np.pad(y, (n // 2, n - 1 - n // 2), mode="edge")
    return np.convolve(ypad, k, mode="valid")


def linfit(t, y):
    """Least squares line y = a + b (t - tm). Returns slope b, its standard error, value at tm, residual SD, tm, R^2."""
    t = np.asarray(t, float)
    y = np.asarray(y, float)
    n = t.size
    tm = t.mean()
    X = t - tm
    Sxx = np.sum(X * X)
    b = np.sum(X * (y - y.mean())) / Sxx
    a = y.mean()
    res = y - (a + b * X)
    s = math.sqrt(np.sum(res ** 2) / (n - 2)) if n > 2 else np.nan
    sb = s / math.sqrt(Sxx) if n > 2 else np.nan
    r2 = 1 - np.sum(res ** 2) / np.sum((y - y.mean()) ** 2)
    return b, sb, a, s, tm, r2


def propagate(f, values, uncs, step=1e-6):
    """Value of f(**values) and its uncertainty for independent inputs, plus the size of every term."""
    f0 = f(**values)
    contrib = {}
    for key, dx in uncs.items():
        if dx is None or dx == 0 or not np.isfinite(dx):
            contrib[key] = 0.0
            continue
        h = abs(values[key]) * step if values[key] != 0 else step
        vp, vm = dict(values), dict(values)
        vp[key] += h
        vm[key] -= h
        contrib[key] = abs((f(**vp) - f(**vm)) / (2 * h) * dx)
    return f0, math.sqrt(sum(c * c for c in contrib.values())), contrib


def read_iolab_csv(path):
    """Read one iOLab CSV export. Returns (t, [channels], column_names).
    Handles the iOLab layout (a 'time' column and 'cal[0]', 'cal[1]', 'cal[2]' columns),
    comma or semicolon separators, and decimal point or decimal comma."""
    path = Path(path)
    with open(path, "r", encoding="utf-8-sig", errors="replace") as fh:
        lines = [fh.readline() for _ in range(40)]
    hdr = 0
    for i, ln in enumerate(lines):
        if re.search(r"(?i)\btime\b|cal\s*\[", ln):
            hdr = i
            break
    first = lines[hdr]
    if first.count(";") > first.count(","):
        sep = ";"
    elif first.count("\t") > first.count(","):
        sep = "\t"
    else:
        sep = ","
    df = pd.read_csv(path, sep=sep, skiprows=hdr, dtype=str, encoding="utf-8-sig", engine="python")
    df.columns = [str(c).strip() for c in df.columns]

    def num(col):
        s = df[col].astype(str).str.strip()
        if sep != ",":
            s = s.str.replace(",", ".", regex=False)
        return pd.to_numeric(s, errors="coerce").to_numpy(dtype=float)

    cols = list(df.columns)
    tcol = next((c for c in cols if c.lower() == "time"), None) \
        or next((c for c in cols if re.match(r"(?i)^time\b", c) and "stamp" not in c.lower()), None) \
        or next((c for c in cols if re.match(r"(?i)^t(\s*\(s\))?$", c)), None) \
        or next((c for c in cols if "time" in c.lower()), None)
    if tcol is None:
        raise ValueError(f"{path.name}: no time column found. Columns: {cols}")
    cal = [c for c in cols if re.match(r"(?i)^cal\s*\[\s*\d+\s*\]", c)]
    cal.sort(key=lambda c: int(re.findall(r"\d+", c)[0]))
    if not cal:
        skip = re.compile(r"(?i)(^index$|^idx$|^unnamed|raw|frame|stamp|^sample|^#)")
        cal = [c for c in cols if c != tcol and not skip.search(c) and np.isfinite(num(c)).mean() > 0.9]
    t = num(tcol)
    chans = [num(c) for c in cal]
    ok = np.isfinite(t)
    for c in chans:
        ok &= np.isfinite(c)
    t = t[ok]
    chans = [c[ok] for c in chans]
    order = np.argsort(t, kind="stable")
    t = t[order]
    chans = [c[order] for c in chans]
    if t.size > 2 and np.median(np.diff(t)) > 0.5:      # looks like milliseconds
        t = t / 1000.0
    if t.size < 10 or not chans:
        raise ValueError(f"{path.name}: could not read data columns. Columns: {cols}")
    return t, chans, cal


def trapz(y, x):
    """Area under y(x) with the trapezoid rule (works on old and new numpy)."""
    y = np.asarray(y, float)
    x = np.asarray(x, float)
    return float(np.sum(0.5 * (y[1:] + y[:-1]) * np.diff(x)))


def show(df):
    try:
        display(df)
    except NameError:
        print(df.to_string(index=False))


def rate(t):
    return 1.0 / np.median(np.diff(t))


def csv_files(folder):
    return sorted([p for p in Path(folder).glob("*.csv") if not p.name.startswith(".")])


def run_folders(part_dir):
    part_dir = Path(part_dir)
    if not part_dir.exists():
        return []
    runs = [p for p in part_dir.iterdir() if p.is_dir() and csv_files(p)]
    key = lambda p: (int(re.findall(r"\d+", p.name)[0]) if re.findall(r"\d+", p.name) else 10 ** 6, p.name)
    return sorted(runs, key=key)


def is_accelerometer(chans):
    return len(chans) >= 3 and np.nanmedian(np.sqrt(sum(c ** 2 for c in chans[:3]))) > 6.0


def load_gyro(folder):
    """The gyroscope file of one run folder: t and the angular velocity about the rolling axis (offset not removed)."""
    files = csv_files(folder)
    picked = None
    for p in files:
        t, ch, names = read_iolab_csv(p)
        if "gyro" in p.name.lower() or (len(ch) >= 3 and not is_accelerometer(ch)):
            picked = (t, ch, p.name)
            break
    if picked is None:
        raise ValueError(f"{folder}: no gyroscope CSV found among {[p.name for p in files]}")
    t, ch, name = picked
    ch = [c * GYRO_FACTOR for c in ch]
    if GYRO_AXIS in ("x", "y", "z"):
        k = "xyz".index(GYRO_AXIS)
    else:
        k = int(np.argmax([np.nanmax(np.abs(c - np.median(c[:20]))) for c in ch[:3]]))
    note = "" if k == 1 else f" (rolling axis found on {'xyz'[k]}, not y: check how the rolls sit)"
    return t, ch[k], name, "xyz"[k], note


def load_acc(folder):
    for p in csv_files(folder):
        t, ch, names = read_iolab_csv(p)
        if is_accelerometer(ch):
            return t, ch[:3], p.name
    raise ValueError(f"{folder}: no accelerometer CSV found")


GYRO_FACTOR = math.pi / 180 if GYRO_UNITS == "deg/s" else 1.0


def offset_from_start(t, w):
    base = t < t[0] + BASELINE_S
    return float(np.mean(w[base])), float(np.std(w[base], ddof=1))


print("Helpers loaded. Output folder:", OUT_DIR.resolve())
print("Data folder:", DATA_DIR.resolve())
for sub in ["calibration", "newton", "energy"]:
    print(f"  {sub:12s} runs found:", [p.name for p in run_folders(DATA_DIR / sub)])
for sub in ["angle", "table"]:
    print(f"  {sub:12s} files found:", [p.name for p in csv_files(DATA_DIR / sub)] if (DATA_DIR / sub).exists() else [])
if "example" in str(DATA_DIR):
    print("\nNOTE: this is the SIMULATED example data. In the lab set DATA_DIR = Path('data') and type in your own numbers.")

## Step 1: the parts

Each tape roll is a thick hollow cylinder with inner radius $r$ (inside of the cardboard core) and outer radius $R_j$:
$$I_j=\tfrac12\,m_j\left(r_j^2+R_j^2\right),\qquad R_j=\frac{C_j}{2\pi},\qquad r_j=\frac{D_j}{2}.$$
The two rolls together give $I_{rolls}=I_1+I_2$. The model treats the cardboard core as if it were tape. The core is lighter, so the real $I_{rolls}$ is about 1 % larger; the two layer check at the end of this cell shows how much.

For the iOLab a box model gives the range to expect: all mass spread evenly through the box, $I=\tfrac1{12}m(w^2+t^2)$, or all mass in the walls, $I=\tfrac1{12}m(w+t)^2$.

In [ ]:
# ============================== STEP 1: masses, radii, tape rolls ==============================
def roll_I(m, D, C):
    r, R_ = D / 2, C / (2 * math.pi)
    return 0.5 * m * (r ** 2 + R_ ** 2)


rows = []
I_rolls, var_rolls = 0.0, 0.0
for j, (m_j, D_j, C_j) in enumerate(zip(m_roll, D_in, C_out), start=1):
    val, unc, con = propagate(lambda m, D, C: roll_I(m, D, C), dict(m=m_j, D=D_j, C=C_j),
                              dict(m=dm_roll, D=dD_in, C=dC_out))
    I_rolls += val
    var_rolls += unc ** 2
    rows.append(dict(roll=j, m_kg=m_j, r_mm=1000 * D_j / 2, R_mm=1000 * C_j / (2 * math.pi),
                     dR_mm=1000 * dC_out / (2 * math.pi), I=val, dI=unc))
dI_rolls = math.sqrt(var_rolls)
parts = pd.DataFrame(rows)
show(parts.round(6))

R_geo = float(np.mean([C / (2 * math.pi) for C in C_out]))
dR_geo = dC_out / (2 * math.pi) / math.sqrt(len(C_out))
M_sum = m_iolab + sum(m_roll)
dM_sum = math.sqrt(dm_iolab ** 2 + len(m_roll) * dm_roll ** 2)
if M_body is not None:
    M, dM = M_body, dM_body
    print(f"Body weighed whole: M = {fmt(M, dM, 'kg')}; sum of parts {fmt(M_sum, dM_sum, 'kg')} "
          f"(z = {zscore(M, dM, M_sum, dM_sum):.1f})")
else:
    M, dM = M_sum, dM_sum
    print(f"M = sum of parts = {fmt(M, dM, 'kg')}")
print(f"Tape rolls: I_rolls = {fmt_e(I_rolls, dI_rolls, 'kg m^2')}")
print(f"Outer radius from the circumference: R_geo = {fmt(1000 * R_geo, 1000 * dR_geo, 'mm')}"
      f"  (roll 1 and roll 2 differ by {1000 * abs(C_out[0] - C_out[1]) / (2 * math.pi):.2f} mm)")

I_box = m_iolab * (iolab_w ** 2 + iolab_t ** 2) / 12
I_shell = m_iolab * (iolab_w + iolab_t) ** 2 / 12
print(f"iOLab box model: uniform box {I_box:.3e} kg m^2, all mass in the walls {I_shell:.3e} kg m^2")

if CORE_WALL:
    I2 = 0.0
    for m_j, D_j, C_j in zip(m_roll, D_in, C_out):
        r, rc, R_ = D_j / 2, D_j / 2 + CORE_WALL, C_j / (2 * math.pi)
        mc = 750.0 * math.pi * (rc ** 2 - r ** 2) * 0.048          # cardboard about 750 kg/m^3, tape width 48 mm
        I2 += 0.5 * mc * (r ** 2 + rc ** 2) + 0.5 * (m_j - mc) * (rc ** 2 + R_ ** 2)
    print(f"Two layer check (core {1000 * CORE_WALL:.1f} mm of cardboard): I_rolls = {I2:.4e}, "
          f"{(I2 / I_rolls - 1) * 100:+.1f} % against the uniform model, a shift of {I2 - I_rolls:+.1e} kg m^2")
    d_model = abs(I2 - I_rolls)
else:
    d_model = 0.01 * I_rolls
print(f"Model uncertainty of the tape rolls (core lighter than tape), used in the budget: {d_model:.1e} kg m^2")
parts.to_csv(OUT_DIR / "parts_table.csv", index=False)

## Step 2: gyroscope calibration roll

The body is rolled by hand on the flat table through exactly $N$ full turns, from a pen line on the roll touching a mark on the table to the same pen line touching a second mark. The true angle is $2\pi N$, and the distance between the marks is $d$. The gyroscope gives $\Phi=\int\omega\,dt$ (the area under the curve, after the offset is removed). Then
$$k=\frac{\Phi}{2\pi N},\qquad R=\frac{d}{2\pi N}.$$
$k$ is the scale factor of the gyroscope; a perfect sensor has $k=1$. Every gyroscope reading is divided by $k$ from here on. $R$ is the rolling radius, measured where it matters: where the roll touches the surface.

In [ ]:
# ============================== STEP 2: calibration rolls ==============================
cal_rows = []
for i, folder in enumerate(run_folders(DATA_DIR / "calibration")):
    try:
        t, w, name, ax, note = load_gyro(folder)
        b0, s0 = offset_from_start(t, w)
        end = t > t[-1] - BASELINE_S
        b1 = float(np.mean(w[end]))
        b = 0.5 * (b0 + b1)
        Phi = abs(trapz(w - b, t))
        T = t[-1] - t[0]
        dPhi = math.sqrt((0.5 * abs(b1 - b0) * T) ** 2 + (s0 / math.sqrt(max(1, np.sum(t < t[0] + BASELINE_S))) * T) ** 2)
        d = CAL_DIST[i] if i < len(CAL_DIST) else np.nan
        R_j = R_geo if not np.isfinite(d) else d / (2 * math.pi * CAL_TURNS)
        dk = math.sqrt(dPhi ** 2 + (2 * ALIGN_MM / 1000 / R_j) ** 2 / 2) / (2 * math.pi * CAL_TURNS)
        cal_rows.append(dict(roll=folder.name, f_Hz=round(rate(t)), Phi_rad=Phi, turns=Phi / (2 * math.pi),
                             k=Phi / (2 * math.pi * CAL_TURNS), dk=dk, d_m=d, R_mm=1000 * R_j,
                             dR_mm=1000 * dCAL_DIST / (2 * math.pi * CAL_TURNS), offset=b, axis=ax))
        if note:
            print(folder.name, note)
    except Exception as e:
        print(f"{folder.name}: skipped ({e})")
for i, d_ in enumerate(MANUAL_CAL):
    Phi = abs(d_["area"] - d_.get("offset", 0.0) * d_["width"])
    R_j = d_["d"] / (2 * math.pi * CAL_TURNS)
    cal_rows.append(dict(roll=f"manual{i + 1}", f_Hz=np.nan, Phi_rad=Phi, turns=Phi / (2 * math.pi),
                         k=Phi / (2 * math.pi * CAL_TURNS),
                         dk=(2 * ALIGN_MM / 1000 / R_j) / math.sqrt(2) / (2 * math.pi * CAL_TURNS),
                         d_m=d_["d"], R_mm=1000 * R_j, dR_mm=1000 * dCAL_DIST / (2 * math.pi * CAL_TURNS),
                         offset=d_.get("offset", 0.0), axis="y"))

if cal_rows and GYRO_UNITS == "auto" and np.median([c_["k"] for c_ in cal_rows]) > 20:
    GYRO_FACTOR = math.pi / 180
    print("The gyroscope files are in deg/s (k came out near 57). Converting every file to rad/s.")
    for c_ in cal_rows:
        if not str(c_["roll"]).startswith("manual"):
            c_["Phi_rad"] *= GYRO_FACTOR
            c_["turns"] *= GYRO_FACTOR
            c_["k"] *= GYRO_FACTOR
            c_["dk"] *= GYRO_FACTOR
if cal_rows:
    cal = pd.DataFrame(cal_rows)
    show(cal.round(5))
    k_gyro, dk_gyro, n_cal = mean_with_floor(cal["k"], cal["dk"])
    R_roll, dR_roll, _ = mean_with_floor(cal["R_mm"] / 1000, cal["dR_mm"] / 1000)
    print(f"Gyroscope scale factor k = {fmt(k_gyro, dk_gyro)}  ({(k_gyro - 1) * 100:+.2f} %, {plural(n_cal, 'roll')})")
    if n_cal < 3:
        print("  Fewer than 3 rolls: do three, the mean and its SDOM are worth the two extra minutes.")
    print(f"Rolling radius from the rolls R = {fmt(1000 * R_roll, 1000 * dR_roll, 'mm')}; "
          f"from the circumference {fmt(1000 * R_geo, 1000 * dR_geo, 'mm')}: "
          f"z = {zscore(R_roll, dR_roll, R_geo, dR_geo):.1f}, {verdict(zscore(R_roll, dR_roll, R_geo, dR_geo))}")
    cal.to_csv(OUT_DIR / "calibration_table.csv", index=False)
else:
    k_gyro, dk_gyro = 1.0, 0.03
    R_roll, dR_roll = R_geo, dR_geo
    print("No calibration rolls found. Using k = 1.00 ± 0.03 (typical sensor tolerance). This will dominate the result.")

if R_SOURCE == "roll" and cal_rows:
    R, dR = R_roll, dR_roll
else:
    R, dR = R_geo, dR_geo
print(f"R used from here on: {fmt(1000 * R, 1000 * dR, 'mm')} ({'calibration roll' if (R_SOURCE == 'roll' and cal_rows) else 'circumference'})")

## Step 3: the ramp angle

The angle is the input that matters most (see the budget at the end), so it is measured against gravity with the accelerometer, using one axis only.

1. **Reference.** The bare iOLab is held upright against a wall, $y$ straight up, then straight down. The $y$ axis reads $A_{y}=+G+b_y$ and $-G+b_y$, where $G$ is $g$ as this axis sees it (scale error included) and $b_y$ is its offset. So $2G=A_{y,\uparrow}-A_{y,\downarrow}$.
2. **Slope.** The iOLab lies on its back on the board, $y$ pointing up the slope, then turned around so $y$ points down the slope: $A_y=+G\sin\theta+b_y$ and $-G\sin\theta+b_y$.

Dividing the two differences removes the offset and the scale error of the axis:
$$\sin\theta=\frac{A_{y,up}-A_{y,down}}{A_{y,\uparrow}-A_{y,\downarrow}}.$$
This angle is measured against gravity, so a tilted table does not matter. Doing it at three places along the board shows whether the board bends.

The ruler gives the angle against the table top. The same flip on the table measures how much the table itself is tilted, $\tau$, so the ruler angle can be corrected: $\theta=\theta_{ruler}+\tau$.

In [ ]:
# ============================== STEP 3: ramp angle ==============================
VERTICAL_TILT_DEG = 2.0      # deg, how far from vertical the iOLab may have been in the upright reference


def still_segments(t, A, sd_max=0.08, min_len=1.5, trim=0.3):
    """Parts of an accelerometer recording where the iOLab lies still (low noise on all axes for at least min_len s)."""
    f = rate(t)
    n = max(5, int(0.25 * f))
    sds = [np.nan_to_num(pd.Series(c).rolling(n, center=True, min_periods=n // 2).std().to_numpy(), nan=9.0) for c in A]
    quiet = max(np.percentile(s_, 10) for s_ in sds)       # the noise of this sensor when it lies still
    limit = max(sd_max, 2.5 * quiet)
    still = np.ones(t.size, bool)
    for s_ in sds:
        still &= s_ < limit
    segs, i = [], 0
    while i < t.size:
        if still[i]:
            j = i
            while j + 1 < t.size and still[j + 1]:
                j += 1
            if t[j] - t[i] >= min_len:
                k = (t >= t[i] + trim) & (t <= t[j] - trim)
                if k.sum() > 10:
                    m_ = [float(np.mean(c[k])) for c in A]
                    sd = float(np.std(A[1][k], ddof=1))
                    segs.append(dict(t0=t[i] + trim, t1=t[j] - trim, n=int(k.sum()), Ax=m_[0], Ay=m_[1], Az=m_[2],
                                     sdy=sd, dAy=sd / math.sqrt(k.sum()), gabs=float(np.linalg.norm(m_))))
            i = j + 1
        else:
            i += 1
    return segs


def split_segments(segs):
    """Upright reference parts (|Ay| close to |A|) and flat parts (iOLab lying on a surface)."""
    vert = [s_ for s_ in segs if abs(s_["Ay"]) > 0.8 * s_["gabs"]]
    flat = [s_ for s_ in segs if abs(s_["Ay"]) < 0.5 * s_["gabs"]]
    return vert, flat


def flip_pairs(flat):
    out, i = [], 0
    while i + 1 < len(flat):
        a, b = flat[i], flat[i + 1]
        if np.sign(a["Ay"]) != np.sign(b["Ay"]):
            up, dn = (a, b) if a["Ay"] > b["Ay"] else (b, a)
            out.append(dict(Ay_up=up["Ay"], Ay_down=dn["Ay"], dA=math.hypot(up["dAy"], dn["dAy"]),
                            gabs=0.5 * (up["gabs"] + dn["gabs"]), first_up=a["Ay"] > 0))
            i += 2
        else:
            i += 1
    return out


G2 = dG2 = None             # 2G from the upright reference
angle_pairs = []
if (DATA_DIR / "angle").exists() and csv_files(DATA_DIR / "angle"):
    t, A, name = load_acc(DATA_DIR / "angle")
    vert, flat = split_segments(still_segments(t, A))
    ups = [s_ for s_ in vert if s_["Ay"] > 0]
    dns = [s_ for s_ in vert if s_["Ay"] < 0]
    if ups and dns:
        G2 = np.mean([s_["Ay"] for s_ in ups]) - np.mean([s_["Ay"] for s_ in dns])
        dG2 = math.hypot(np.mean([s_["dAy"] for s_ in ups]), np.mean([s_["dAy"] for s_ in dns]))
    angle_pairs = flip_pairs(flat)
    print(f"{name}: {len(vert)} upright parts, {len(flat)} flat parts, {len(angle_pairs)} up/down pairs, rate {rate(t):.0f} Hz")
if MANUAL_ANGLE:
    d_ = MANUAL_ANGLE
    dA = d_.get("sd", 0.03) / math.sqrt(max(1.0, d_.get("f", 100) * d_.get("width", 3.0)))
    if "Ay_vertical_up" in d_:
        G2, dG2 = d_["Ay_vertical_up"] - d_["Ay_vertical_down"], math.sqrt(2) * dA
    for up_, dn_ in d_.get("pairs", []):
        angle_pairs.append(dict(Ay_up=up_, Ay_down=dn_, dA=math.sqrt(2) * dA, gabs=g, first_up=True))

sin_acc = dsin_acc = None
if angle_pairs:
    if G2 is None:
        print("No upright reference found: dividing by |A| instead. The scale error of the axis then stays in (about 1 %).")
    ref = G2 if G2 is not None else None
    for p_ in angle_pairs:
        den = ref if ref is not None else 2 * p_["gabs"]
        p_["sin"] = (p_["Ay_up"] - p_["Ay_down"]) / den
        p_["dsin"] = p_["dA"] / den
        p_["offset_y"] = 0.5 * (p_["Ay_up"] + p_["Ay_down"])
    ang = pd.DataFrame(angle_pairs)
    ang["theta_deg"] = np.degrees(np.arcsin(ang["sin"]))
    show(ang[["Ay_up", "Ay_down", "offset_y", "sin", "dsin", "theta_deg"]].round(5))
    s_mean, s_stat, n_ang = mean_with_floor(ang["sin"], ang["dsin"])
    rel_ref = (math.hypot(dG2 / G2, 1 - math.cos(math.radians(VERTICAL_TILT_DEG))) if G2 is not None else 0.01)
    sin_acc = s_mean
    dsin_acc = math.hypot(s_stat, s_mean * rel_ref)
    th = math.asin(sin_acc)
    if G2 is not None:
        print(f"Upright reference: 2G = {fmt(G2, dG2, 'm/s^2')}, so the y axis reads g as {G2 / 2:.3f} m/s^2 "
              f"({(G2 / 2 / g - 1) * 100:+.2f} % against {g})")
    print(f"Accelerometer: sin(theta) = {fmt(sin_acc, dsin_acc)}, theta = {fmt(math.degrees(th), math.degrees(dsin_acc / math.cos(th)), 'deg')}"
          f"  ({plural(n_ang, 'place')} along the board; the offset of the y axis, {ang['offset_y'].mean():+.3f} m/s^2, cancels in the flip)")
    if n_ang < 3:
        print("  Fewer than 3 places: the uncertainty above is only sensor noise. With 3 places it also covers a board that bends.")
    ang.to_csv(OUT_DIR / "angle_table.csv", index=False)

tau = dtau = 0.0
if (DATA_DIR / "table").exists() and csv_files(DATA_DIR / "table"):
    t, A, name = load_acc(DATA_DIR / "table")
    vt, ft = split_segments(still_segments(t, A))
    tp = flip_pairs(ft)
    if tp:
        p_ = tp[0]
        den = G2 if G2 is not None else 2 * p_["gabs"]
        sgn = 1.0 if p_["first_up"] else -1.0      # first still part: y pointing toward the high end of the ramp
        tau, dtau = sgn * math.asin((p_["Ay_up"] - p_["Ay_down"]) / den), p_["dA"] / den
        print(f"Table tilt along the ramp: tau = {fmt(math.degrees(tau), math.degrees(dtau), 'deg')} "
              f"(positive: the table rises toward the high end of the ramp)")

sin_rul = dsin_rul = None
if RULER:
    H_ = RULER["h_high"] - RULER["h_low"]
    sin_rul = H_ / RULER["L"]
    dsin_rul = sin_rul * math.sqrt((math.sqrt(2) * RULER["dh"] / H_) ** 2 + (RULER["dL"] / RULER["L"]) ** 2)
    th_r = math.asin(sin_rul)
    dth_r = dsin_rul / math.cos(th_r)
    print(f"Ruler: sin(theta) = {fmt(sin_rul, dsin_rul)}, theta = {fmt(math.degrees(th_r), math.degrees(dth_r), 'deg')} against the table top")
    if sin_acc is not None:
        th_a = math.asin(sin_acc)
        dth_a = dsin_acc / math.cos(th_a)
        z1 = zscore(th_r, dth_r, th_a, dth_a)
        print(f"  ruler against accelerometer: z = {z1:.1f}, {verdict(z1)}")
        if tau:
            z2 = zscore(th_r + tau, math.hypot(dth_r, dtau), th_a, dth_a)
            print(f"  ruler + table tilt = {fmt(math.degrees(th_r + tau), math.degrees(math.hypot(dth_r, dtau)), 'deg')}: "
                  f"z = {z2:.1f}, {verdict(z2)}")

if ANGLE_SOURCE == "ruler" or (ANGLE_SOURCE == "auto" and sin_acc is None):
    th_r = math.asin(sin_rul) + tau
    sin_th, dsin_th = math.sin(th_r), math.hypot(dsin_rul, math.cos(th_r) * dtau)
    angle_used = "ruler" + (" + table tilt" if tau else "")
else:
    sin_th, dsin_th = sin_acc, dsin_acc
    angle_used = "accelerometer"
theta = math.asin(sin_th)
print(f"\nAngle used from here on ({angle_used}): sin(theta) = {fmt(sin_th, dsin_th)}, "
      f"theta = {fmt(math.degrees(theta), math.degrees(dsin_th / math.cos(theta)), 'deg')}")

## Step 4: Method N, rotational Newton's second law (up and down runs)

Rolling resistance always opposes the motion, so it points down the ramp while the body rolls up and up the ramp while it rolls down. About the contact line $P$, with the angular acceleration counted positive in the rolling down sense:
$$I_P\,\alpha_{up}=MgR\sin\theta+F_rR,\qquad I_P\,\alpha_{down}=MgR\sin\theta-F_rR.$$
The mean $\bar\alpha=(\alpha_{up}+\alpha_{down})/2$ has no friction in it:
$$I_P=\frac{MgR\sin\theta}{\bar\alpha},\qquad I=I_P-MR^2,\qquad I_{iOLab}=I-I_{rolls}.$$
The difference gives the size of the losses, $\varepsilon=(\alpha_{up}-\alpha_{down})/(\alpha_{up}+\alpha_{down})$, the friction force as a fraction of $Mg\sin\theta$. Method E needs it.

What the code does for every run: it removes the gyroscope offset (the still first half second), turns the sign so the push up the ramp is negative, finds the end of the push (most negative $\omega$), the turning point ($\omega=0$) and the catch (largest $\omega$ before the hand stops it), fits straight lines to $\omega(t)$ in the two windows, and divides the slopes by $k$.

In [ ]:
# ============================== STEP 4: Newton runs ==============================
rho_air = 1.20
k_drag = 0.5 * rho_air * C_D * (2 * R * BODY_LENGTH)        # F_drag = k_drag v^2


def analyse_newton(folder):
    t, w, name, ax, note = load_gyro(folder)
    b, sb = offset_from_start(t, w)
    w = w - b
    f = rate(t)
    big = np.nonzero(np.abs(w) > 3.0)[0]
    if big.size == 0:
        raise ValueError("no motion found")
    w = w if w[big[0]] < 0 else -w                   # push up the ramp = negative
    ws = moving_average(w, max(1, int(SMOOTH_MS / 1000 * f)))
    i0 = big[0]
    i_zero = next((i for i in range(i0, len(ws) - 1) if ws[i] < 0 <= ws[i + 1]), None)
    if i_zero is None:
        raise ValueError("no turning point (omega never crosses zero)")
    i_min = i0 + int(np.argmin(ws[i0:i_zero]))
    t_turn = t[i_zero] - ws[i_zero] * (t[i_zero + 1] - t[i_zero]) / (ws[i_zero + 1] - ws[i_zero])
    i_max = i_zero + int(np.argmax(ws[i_zero:]))
    t_push, t_catch = t[i_min], t[i_max]
    win = MANUAL_WINDOWS.get(folder.name, {})
    up = win.get("up", (t_push + MARGIN_PUSH_S, t_turn - MARGIN_TURN_S))
    dn = win.get("down", (t_turn + MARGIN_TURN_S, t_catch - MARGIN_END_S))
    ku = (t >= up[0]) & (t <= up[1])
    kd = (t >= dn[0]) & (t <= dn[1])
    if ku.sum() < 20 or kd.sum() < 20:
        raise ValueError("windows too short; set MANUAL_WINDOWS for this run")
    au, sau, _, resu, _, r2u = linfit(t[ku], w[ku])
    ad, sad, _, resd, _, r2d = linfit(t[kd], w[kd])
    v2u = float(np.mean((w[ku] / k_gyro * R) ** 2))
    v2d = float(np.mean((w[kd] / k_gyro * R) ** 2))
    return dict(run=folder.name, f=f, t=t, w=w, up=up, dn=dn, t_turn=t_turn, t_push=t_push, t_catch=t_catch,
                a_up=au / k_gyro, da_up=sau / k_gyro, a_dn=ad / k_gyro, da_dn=sad / k_gyro, v2u=v2u, v2d=v2d,
                w_up=up[1] - up[0], w_dn=dn[1] - dn[0], w_max=float(np.max(np.abs(w))) / k_gyro, note=note, offset=b)


def signed_budget(f, values, uncs, step=1e-6):
    """f(**values) and the signed terms (df/dx) * dx for every input with an uncertainty."""
    f0 = f(**values)
    terms = {}
    for key, dx in uncs.items():
        if not dx or not np.isfinite(dx):
            continue
        h = abs(values[key]) * step if values[key] != 0 else step
        vp, vm = dict(values), dict(values)
        vp[key] += h
        vm[key] -= h
        terms[key] = (f(**vp) - f(**vm)) / (2 * h) * dx
    return f0, terms


def total(terms):
    return math.sqrt(sum(v * v for v in terms.values()))


def I_newton(M, R, s, a_meas, k, I_rolls, model, drag):
    """I_iOLab from Method N. a_meas = mean angular acceleration as the gyroscope reads it (before dividing by k)."""
    IP = M * g * R * s * k / a_meas
    return IP - M * R ** 2 - I_rolls - model - drag


newton = []
for folder in run_folders(DATA_DIR / "newton"):
    try:
        newton.append(analyse_newton(folder))
    except Exception as e:
        print(f"{folder.name}: skipped ({e})")
for i, d_ in enumerate(MANUAL_NEWTON):
    newton.append(dict(run=f"manual{i + 1}", a_up=abs(d_["s_up"]) / k_gyro, da_up=np.nan, a_dn=abs(d_["s_down"]) / k_gyro,
                       da_dn=np.nan, v2u=np.nan, v2d=np.nan, w_up=np.nan, w_dn=np.nan,
                       w_max=abs(d_.get("w_max", np.nan)) / k_gyro, note=""))

if newton:
    for r in newton:
        r["a_bar"] = 0.5 * (r["a_up"] + r["a_dn"])
        r["eps"] = (r["a_up"] - r["a_dn"]) / (r["a_up"] + r["a_dn"])
        r["IP"] = M * g * R * sin_th / r["a_bar"]
        r["I_iolab"] = r["IP"] - M * R ** 2 - I_rolls
    tabN = pd.DataFrame([{k_: r[k_] for k_ in ["run", "a_up", "da_up", "a_dn", "da_dn", "a_bar", "eps", "IP", "I_iolab",
                                               "w_up", "w_dn", "w_max"]} for r in newton])
    show(tabN.round(5))
    a_up_m, da_up_m, nN = mean_with_floor(tabN["a_up"], tabN["da_up"])
    a_dn_m, da_dn_m, _ = mean_with_floor(tabN["a_dn"], tabN["da_dn"])
    a_bar, da_bar, _ = mean_with_floor(tabN["a_bar"], 0.5 * np.hypot(tabN["da_up"], tabN["da_dn"]))
    eps, deps, _ = mean_with_floor(tabN["eps"], [0.0])
    have_v = any(np.isfinite(r["v2u"]) for r in newton)
    v2u_m = np.nanmean([r["v2u"] for r in newton]) if have_v else np.nan
    v2d_m = np.nanmean([r["v2d"] for r in newton]) if have_v else np.nan
    if not have_v:                      # manual mode: mean v^2 in a window of a straight line from v_max to 0 is v_max^2 / 3
        wm = [r.get("w_max", np.nan) for r in newton]
        if np.isfinite(wm).any():
            v2u_m = v2d_m = (np.nanmean(wm) * R) ** 2 / 3
    if not np.isfinite(v2u_m):
        v2u_m = v2d_m = 0.0
    f_N = k_drag * 0.5 * (v2u_m + v2d_m) / (M * g * sin_th)          # drag share already inside epsilon
    IP_N = M * g * R * sin_th / a_bar
    drag_N = IP_N * k_drag * abs(v2u_m - v2d_m) / (2 * M * g * sin_th)  # what drag leaves in the mean of the two
    print(f"alpha_up   = {fmt(a_up_m, da_up_m, 'rad/s^2')}   ({plural(nN, 'run')}, mean ± SDOM, after dividing by k)")
    if nN < 3:
        print("  Fewer than 3 runs: the uncertainty is only the fit uncertainty, which is too small. Do 6 to 8 runs.")
    print(f"alpha_down = {fmt(a_dn_m, da_dn_m, 'rad/s^2')}")
    print(f"mean       = {fmt(a_bar, da_bar, 'rad/s^2')}")
    print(f"epsilon = (alpha_up - alpha_down)/(alpha_up + alpha_down) = {fmt(eps, deps)}  ->  "
          f"loss force F = {fmt(eps * M * g * sin_th, deps * M * g * sin_th, 'N')} (rolling resistance plus a little air drag)")
    tabN.to_csv(OUT_DIR / "newton_table.csv", index=False)

    valsN = dict(M=M, R=R, s=sin_th, a_meas=a_bar * k_gyro, k=k_gyro, I_rolls=I_rolls, model=0.0, drag=0.0)
    uncN = dict(M=dM, R=dR, s=dsin_th, a_meas=da_bar * k_gyro, k=dk_gyro, I_rolls=dI_rolls, model=d_model, drag=drag_N)
    I_N, termsN = signed_budget(I_newton, valsN, uncN)
    dI_N = total(termsN)
    dIP_N = IP_N * math.sqrt((dM / M) ** 2 + (dR / R) ** 2 + (dsin_th / sin_th) ** 2 + (da_bar / a_bar) ** 2 + (dk_gyro / k_gyro) ** 2)
    I_body_N = IP_N - M * R ** 2
    dI_body_N = math.sqrt(max(0.0, dI_N ** 2 - termsN["I_rolls"] ** 2 - termsN.get("model", 0) ** 2))
    print(f"\nI_P (about the contact line) = {fmt_e(IP_N, dIP_N, 'kg m^2', -3)}")
    print(f"I (whole body) = I_P - M R^2 = {fmt_e(I_body_N, dI_body_N, 'kg m^2', -4)},  I/(M R^2) = {I_body_N / (M * R ** 2):.4f}")
    print(f"METHOD N: I_iOLab = I - I_rolls = {fmt_e(I_N, dI_N, 'kg m^2')}")
    print("  budget, size of each term (kg m^2):", ", ".join(f"{k_} {abs(v):.1e}" for k_, v in sorted(termsN.items(), key=lambda x: -abs(x[1]))))
else:
    print("No Newton runs found.")
    eps, deps, f_N = 0.0, 0.0, 0.0

### Plots of the Newton runs

Check every run: the up window must start after the push has ended and stop before the turning point; the down window must stop before the hand catches the body. The fitted lines (black) should follow the data.

In [ ]:
def plot_newton(r, ax=None, title=True):
    ax = ax or plt.gca()
    ax.plot(r["t"], r["w"], color=BLUE, lw=0.8, label=r"$\omega$ (gyroscope, down ramp positive)")
    for (a, b), lab, col in [(r["up"], "up window", ORANGE), (r["dn"], "down window", AQUA)]:
        ax.axvspan(a, b, color=col, alpha=0.15, lw=0)
        k = (r["t"] >= a) & (r["t"] <= b)
        bb, _, aa, _, tm, _ = linfit(r["t"][k], r["w"][k])
        tt = np.array([a, b])
        ax.plot(tt, aa + bb * (tt - tm), color=INK, lw=1.0)
    ax.axhline(0, color=MUTED, lw=0.6)
    ax.axvline(r["t_turn"], color=MUTED, lw=0.6, ls="--")
    ax.set_xlabel("time t (s)")
    ax.set_ylabel(r"$\omega$ (rad/s)")
    if title:
        ax.set_title(f"{r['run']}: alpha_up = {r['a_up']:.3f}, alpha_down = {r['a_dn']:.3f} rad/s^2", fontsize=9)


csv_newton = [r for r in newton if "t" in r]
if csv_newton:
    n_ = len(csv_newton)
    fig, axs = plt.subplots((n_ + 1) // 2, 2, figsize=(10, 2.6 * ((n_ + 1) // 2)), squeeze=False)
    for r, ax in zip(csv_newton, axs.flat):
        plot_newton(r, ax)
        ax.set_xlim(r["t_push"] - 0.4, r["t_catch"] + 0.3)
    for ax in list(axs.flat)[n_:]:
        ax.axis("off")
    fig.tight_layout()
    fig.savefig(OUT_DIR / "newton_all_runs.png", dpi=150)
    plt.show()

## Step 5: Method E, energy (release runs)

The body starts at rest with its contact line at $A$ and rolls the distance $s_{AB}$ along the board to the lower end $B$. It drops $\Delta h=s_{AB}\sin\theta$. With rolling without slipping the kinetic energy is $\tfrac12Mv^2+\tfrac12I\omega^2=\tfrac12I_P\,\omega^2$. Rolling resistance takes away the work $\varepsilon\,Mg\,\Delta h$ and air drag a little more, $\Delta f\,Mg\,\Delta h$:
$$Mg\Delta h=\tfrac12 I_P\,\omega_B^2+(\varepsilon+\Delta f)\,Mg\Delta h\quad\Longrightarrow\quad I_P=\frac{2Mg\,s_{AB}\sin\theta\,(1-\varepsilon-\Delta f)}{\omega_B^2}.$$
$\Delta f$ is a small air drag correction. Drag grows with $v^2$, and the energy runs end faster than the average speed in the Newton runs, so they lose a little more to drag than $\varepsilon$ contains: $\Delta f=k_dv_B^2/(2Mg\sin\theta)-k_d\langle v^2\rangle_N/(Mg\sin\theta)$ with $k_d=\tfrac12\rho C_DA$. It is an estimate, so half of it is used as its uncertainty.

Finding $\omega_B$: when the contact line reaches the end of the board, the body leaves the board. In the short flight no torque acts, so $\omega$ stays constant until the body lands on the table. The code fits a hinge to the last part of the rise: a straight line up to the corner time $t_B$, flat after it. The value at the corner is $\omega_B$. Check the zoom plots: the corner must sit where the rise stops.

Two checks come for free: the slope of the rise is $\alpha_{down}$ (compare with Method N), and the gyroscope angle from release to $t_B$ times $R$ must equal $s_{AB}$ (no slipping, right distances).

In [ ]:
# ============================== STEP 5: energy runs ==============================
def hinge_fit(t, w, i_a, i_b):
    """Fit w = c + a (min(t, tB) - tB) on samples i_a..i_b. Returns tB, c (= omega at the corner), a, sigma_c, sse."""
    tt, ww = t[i_a:i_b + 1], w[i_a:i_b + 1]
    best = None
    for j in range(5, len(tt) - 3):
        for frac in (0.0, 0.25, 0.5, 0.75):
            tB = tt[j] + frac * (tt[1] - tt[0])
            X = np.column_stack([np.ones_like(tt), np.minimum(tt, tB) - tB])
            coef, res, *_ = np.linalg.lstsq(X, ww, rcond=None)
            sse = float(np.sum((ww - X @ coef) ** 2))
            if best is None or sse < best[4]:
                best = (tB, coef[0], coef[1], X, sse)
    tB, c, a, X, sse = best
    s2 = sse / (len(tt) - 2)
    cov = s2 * np.linalg.inv(X.T @ X)
    return tB, c, a, math.sqrt(cov[0, 0]), sse


def analyse_energy(folder):
    t, w, name, ax, note = load_gyro(folder)
    b, sb = offset_from_start(t, w)
    w = w - b
    f = rate(t)
    big = np.nonzero(np.abs(w) > 3.0)[0]
    if big.size == 0:
        raise ValueError("no motion found")
    w = w if w[big[0]] > 0 else -w                   # rolling down the ramp = positive
    ws = moving_average(w, max(1, int(SMOOTH_MS / 1000 * f)))
    i_pk = int(np.argmax(ws))
    lev = max(0.15, 3 * sb)                          # release: last time before the rise with omega near zero
    i_r = i_pk
    while i_r > 0 and ws[i_r] > lev:
        i_r -= 1
    t_rel = t[i_r]
    drop = max(5 * sb, 0.004 * ws[i_pk], 0.05)       # landing: first clear drop after the peak
    i_land = next((i for i in range(i_pk, len(ws)) if ws[i] < ws[i_pk] - drop), min(len(ws) - 1, i_pk + int(0.03 * f)))
    if folder.name in MANUAL_CORNER:
        tB = MANUAL_CORNER[folder.name]
        kk = (t >= tB - 0.25) & (t <= tB)
        a_, sa_, c0, res_, tm_, _ = linfit(t[kk], w[kk])
        wB, dwB, a_hinge = c0 + a_ * (tB - tm_), res_ / math.sqrt(kk.sum()), a_
    else:
        i_a = int(np.searchsorted(t, max(t_rel + 0.15, t[i_pk] - 0.30)))
        i_b = max(i_a + 20, i_land - 2)
        tB, wB, a_hinge, dwB, _ = hinge_fit(t, w, i_a, i_b)
    kr = (t >= t_rel + 0.12) & (t <= tB - 0.05)      # slope over the long part of the descent: alpha_down check
    a_long, sa_long, _, _, _, _ = linfit(t[kr], w[kr])
    kk = (t >= t_rel - 0.05) & (t < tB)             # gyroscope distance from release to the corner
    s_gyro = R * trapz(np.append(w[kk], wB), np.append(t[kk], tB)) / k_gyro
    return dict(run=folder.name, f=f, t=t, w=w, t_rel=t_rel, tB=tB, t_land=t[i_land], wB=wB / k_gyro,
                dwB=dwB / k_gyro, a_rise=a_long / k_gyro, da_rise=sa_long / k_gyro, wB_meas=wB, a_hinge=a_hinge,
                s_gyro=s_gyro, flight_ms=1000 * (t[i_land] - tB), note=note, offset=b)


def I_energy(M, R, s, D_gate, w_meas, k, eps, fdrag, I_rolls, model):
    """I_iOLab from Method E. w_meas = omega_B as the gyroscope reads it (before dividing by k)."""
    IP = 2 * M * g * (D_gate + R) * s * (1 - eps - fdrag) * k ** 2 / w_meas ** 2
    return IP - M * R ** 2 - I_rolls - model


energy = []
for folder in run_folders(DATA_DIR / "energy"):
    try:
        energy.append(analyse_energy(folder))
    except Exception as e:
        print(f"{folder.name}: skipped ({e})")
for i, d_ in enumerate(MANUAL_ENERGY):
    energy.append(dict(run=f"manual{i + 1}", wB=(abs(d_["mu"]) + abs(d_["s"]) * d_["width"] / 2) / k_gyro, dwB=np.nan,
                       a_rise=np.nan, da_rise=np.nan, s_gyro=np.nan, flight_ms=np.nan, note=""))

s_AB = D_GATE + R
ds_AB = math.hypot(dD_GATE, dR)
if energy:
    if not newton:
        print("No Newton runs, so epsilon is unknown. Using epsilon = 0 ± 0.02. The result is then too large.")
        eps, deps, f_N = 0.0, 0.02, 0.0
    wB_m, dwB_m, nE = mean_with_floor([r["wB"] for r in energy], [r["dwB"] for r in energy])
    f_E = k_drag * (wB_m * R) ** 2 / (2 * M * g * sin_th)
    fdrag = f_E - f_N
    for r in energy:
        r["IP"] = 2 * M * g * s_AB * sin_th * (1 - eps - fdrag) / r["wB"] ** 2
        r["I_iolab"] = r["IP"] - M * R ** 2 - I_rolls
    tabE = pd.DataFrame([{k_: r[k_] for k_ in ["run", "wB", "dwB", "a_rise", "da_rise", "s_gyro", "flight_ms", "IP",
                                               "I_iolab"]} for r in energy])
    show(tabE.round(5))
    dh = s_AB * sin_th
    print(f"s_AB = D_gate + R = {fmt(s_AB, ds_AB, 'm')},  height drop = s_AB sin(theta) = "
          f"{fmt(dh, dh * math.hypot(ds_AB / s_AB, dsin_th / sin_th), 'm')}")
    print(f"omega_B = {fmt(wB_m, dwB_m, 'rad/s')}  ({plural(nE, 'run')}, mean ± SDOM, after dividing by k), v_B = {wB_m * R:.2f} m/s")
    if nE < 3:
        print("  Fewer than 3 runs: the uncertainty is only the fit uncertainty, which is too small. Do 6 to 8 runs.")
    print(f"air drag: k_d = {k_drag:.4f} kg/m; energy runs lose {100 * f_E:.2f} % of M g dh to drag, epsilon already "
          f"holds {100 * f_N:.2f} %, so the correction is {100 * fdrag:.2f} %")
    if np.isfinite(tabE["s_gyro"]).any():
        sg, dsg, _ = mean_with_floor(tabE["s_gyro"], [0.0])
        z_s = zscore(sg, math.hypot(dsg, sg * dk_gyro / k_gyro), s_AB, ds_AB)
        print(f"check: distance from the gyroscope, R times the angle = {fmt(sg, dsg, 'm')}, against s_AB {fmt(s_AB, ds_AB, 'm')}: "
              f"z = {z_s:.1f}, {verdict(z_s)}")
    if newton and np.isfinite(tabE["a_rise"]).any():
        ar, dar, _ = mean_with_floor(tabE["a_rise"], tabE["da_rise"])
        z_a = zscore(ar, dar, a_dn_m, da_dn_m)
        print(f"check: slope of the release rise {fmt(ar, dar, 'rad/s^2')} against alpha_down of Method N "
              f"{fmt(a_dn_m, da_dn_m, 'rad/s^2')}: z = {z_a:.1f}, {verdict(z_a)} "
              f"(the release runs are faster, so a little more drag makes their slope a bit smaller)")
    tabE.to_csv(OUT_DIR / "energy_table.csv", index=False)

    valsE = dict(M=M, R=R, s=sin_th, D_gate=D_GATE, w_meas=wB_m * k_gyro, k=k_gyro, eps=eps, fdrag=fdrag,
                 I_rolls=I_rolls, model=0.0)
    uncE = dict(M=dM, R=dR, s=dsin_th, D_gate=dD_GATE, w_meas=dwB_m * k_gyro, k=dk_gyro, eps=deps,
                fdrag=abs(fdrag) * dC_D / C_D, I_rolls=dI_rolls, model=d_model)
    I_E, termsE = signed_budget(I_energy, valsE, uncE)
    dI_E = total(termsE)
    IP_E = 2 * M * g * s_AB * sin_th * (1 - eps - fdrag) / wB_m ** 2
    dIP_E = IP_E * math.sqrt((dM / M) ** 2 + (ds_AB / s_AB) ** 2 + (dsin_th / sin_th) ** 2 + (2 * dwB_m / wB_m) ** 2
                             + (deps / (1 - eps)) ** 2 + (2 * dk_gyro / k_gyro) ** 2)
    I_body_E = IP_E - M * R ** 2
    dI_body_E = math.sqrt(max(0.0, dI_E ** 2 - termsE["I_rolls"] ** 2 - termsE.get("model", 0) ** 2))
    print(f"\nI_P (about the contact line) = {fmt_e(IP_E, dIP_E, 'kg m^2', -3)}")
    print(f"I (whole body) = I_P - M R^2 = {fmt_e(I_body_E, dI_body_E, 'kg m^2', -4)},  I/(M R^2) = {I_body_E / (M * R ** 2):.4f}")
    print(f"METHOD E: I_iOLab = I - I_rolls = {fmt_e(I_E, dI_E, 'kg m^2')}")
    print("  budget, size of each term (kg m^2):", ", ".join(f"{k_} {abs(v):.1e}" for k_, v in sorted(termsE.items(), key=lambda x: -abs(x[1]))))
else:
    print("No energy runs found.")

### Plots of the energy runs

Left: the whole run. Right: zoom on the end of the board. The orange line is the hinge fit; the dot is $(t_B,\omega_B)$. After the corner the body flies for a few hundredths of a second (flat part), then lands on the table (small drop).

In [ ]:
def plot_energy(r, ax_full=None, ax_zoom=None):
    if ax_full is not None:
        ax_full.plot(r["t"], r["w"], color=BLUE, lw=0.8)
        ax_full.axvline(r["t_rel"], color=MUTED, lw=0.6, ls="--")
        ax_full.axvline(r["tB"], color=ORANGE, lw=0.8)
        ax_full.set_xlabel("time t (s)")
        ax_full.set_ylabel(r"$\omega$ (rad/s)")
        ax_full.set_title(f"{r['run']}: omega_B = {r['wB']:.3f} rad/s", fontsize=9)
    if ax_zoom is not None:
        k = (r["t"] > r["tB"] - 0.25) & (r["t"] < r["tB"] + 0.12)
        ax_zoom.plot(r["t"][k], r["w"][k], ".", ms=2.5, color=BLUE)
        tt = np.linspace(r["tB"] - 0.25, min(r["tB"] + 0.12, r["t_land"]), 200)
        ax_zoom.plot(tt, r["wB_meas"] + r["a_hinge"] * (np.minimum(tt, r["tB"]) - r["tB"]), color=ORANGE, lw=1)
        ax_zoom.plot([r["tB"]], [r["wB_meas"]], "o", color=ORANGE, ms=5)
        ax_zoom.set_xlabel("time t (s), zoom")


csv_energy = [r for r in energy if "t" in r]
if csv_energy:
    n_ = len(csv_energy)
    fig, axs = plt.subplots(n_, 2, figsize=(10, 2.3 * n_), squeeze=False)
    for r, (a1, a2) in zip(csv_energy, axs):
        plot_energy(r, a1, a2)
        a1.set_xlim(r["t_rel"] - 0.3, r["t_land"] + 0.4)
    fig.tight_layout()
    fig.savefig(OUT_DIR / "energy_all_runs.png", dpi=150)
    plt.show()

## Step 6: results, comparison and report figures

Both methods use the same $M$, $R$, $\sin\theta$, $k$ and $I_{rolls}$. An error in $\sin\theta$, $M$ or $I_{rolls}$ moves both results by the same amount, so it cannot make them disagree. The agreement test therefore uses the uncertainty of the difference $I_N-I_E$, where those shared terms cancel and only the parts that differ remain (the run to run scatter, $s_{AB}$, $\varepsilon$, and the part of $R$ and $k$ that enters the two formulas differently). The combined value is the mean weighted by the independent parts; its uncertainty keeps the shared terms.

In [ ]:
# ============================== STEP 6: results ==============================
lines = []
if newton and energy:
    keys = sorted(set(termsN) | set(termsE))
    shared = [k_ for k_ in keys if k_ in termsN and k_ in termsE]
    d_diff = math.sqrt(sum((termsN.get(k_, 0.0) - termsE.get(k_, 0.0)) ** 2 for k_ in keys))
    indN = math.sqrt(sum(v ** 2 for k_, v in termsN.items() if k_ not in shared))
    indE = math.sqrt(sum(v ** 2 for k_, v in termsE.items() if k_ not in shared))
    wN, wE = 1 / indN ** 2, 1 / indE ** 2
    I_comb = (wN * I_N + wE * I_E) / (wN + wE)
    d_comb = math.sqrt(sum(((wN * termsN.get(k_, 0.0) + wE * termsE.get(k_, 0.0)) / (wN + wE)) ** 2 for k_ in keys))
    zNE = abs(I_N - I_E) / d_diff
    lines += [f"Method N (torque):  I_iOLab = {fmt_e(I_N, dI_N, 'kg m^2')}",
              f"Method E (energy):  I_iOLab = {fmt_e(I_E, dI_E, 'kg m^2')}",
              f"Difference N - E = {fmt_e(I_N - I_E, d_diff, 'kg m^2')}: z = {zNE:.1f}, {verdict(zNE)}",
              f"Combined: I_iOLab = {fmt_e(I_comb, d_comb, 'kg m^2')}  (weights {wN / (wN + wE):.2f} N, {wE / (wN + wE):.2f} E)",
              f"Box model: uniform {I_box:.2e}, all mass in the walls {I_shell:.2e} kg m^2. "
              f"The combined value is {'inside' if I_box <= I_comb <= I_shell else 'outside'} this range.",
              f"The iOLab is {I_comb / IP_N * 100:.1f} % of I_P, so a 1 % error in I_P is a "
              f"{IP_N / I_comb:.0f} % error in I_iOLab."]
elif newton:
    I_comb, d_comb = I_N, dI_N
    lines += [f"Method N only: I_iOLab = {fmt_e(I_N, dI_N, 'kg m^2')}"]
elif energy:
    I_comb, d_comb = I_E, dI_E
    lines += [f"Method E only: I_iOLab = {fmt_e(I_E, dI_E, 'kg m^2')}"]
print("\n".join(lines))
(OUT_DIR / "results_summary.txt").write_text("\n".join(lines) + "\n", encoding="utf-8")

# Report figure 1: one Newton run and one energy run, side by side
rN = next((r for r in csv_newton if r["run"] == REPORT_RUN_NEWTON), csv_newton[0] if csv_newton else None)
rE = next((r for r in csv_energy if r["run"] == REPORT_RUN_ENERGY), csv_energy[0] if csv_energy else None)
if rN is not None and rE is not None:
    fig = plt.figure(figsize=(7.2, 2.6))
    gs = fig.add_gridspec(1, 3, width_ratios=[1.25, 1.0, 0.75], wspace=0.42)
    a1, a2, a3 = fig.add_subplot(gs[0]), fig.add_subplot(gs[1]), fig.add_subplot(gs[2])
    plot_newton(rN, a1, title=False)
    a1.set_xlim(rN["t_push"] - 0.25, rN["t_catch"] + 0.2)
    a1.set_title(f"Method N, {rN['run']}", fontsize=9)
    yl = a1.get_ylim()
    a1.text(np.mean(rN["up"]), yl[0] + 0.08 * (yl[1] - yl[0]), "up", ha="center", color=ORANGE, fontsize=8)
    a1.text(np.mean(rN["dn"]), yl[0] + 0.08 * (yl[1] - yl[0]), "down", ha="center", color=AQUA, fontsize=8)
    plot_energy(rE, a2, a3)
    a2.set_xlim(rE["t_rel"] - 0.2, rE["t_land"] + 0.3)
    a2.set_title(f"Method E, {rE['run']}", fontsize=9)
    a3.set_title("corner at the board end", fontsize=9)
    fig.savefig(OUT_DIR / "report_figure_runs.png", dpi=300, bbox_inches="tight")
    fig.savefig(OUT_DIR / "report_figure_runs.pdf", bbox_inches="tight")
    plt.show()

# Report figure 2: I_iOLab per run for both methods, and the uncertainty budget
if newton and energy:
    fig, (b1, b2) = plt.subplots(1, 2, figsize=(7.2, 2.6), gridspec_kw=dict(width_ratios=[1.2, 1], wspace=0.35))
    xN = np.arange(1, len(newton) + 1) - 0.12
    xE = np.arange(1, len(energy) + 1) + 0.12
    b1.axhspan((I_N - dI_N) * 1e4, (I_N + dI_N) * 1e4, color=BLUE, alpha=0.12, lw=0)
    b1.axhspan((I_E - dI_E) * 1e4, (I_E + dI_E) * 1e4, color=ORANGE, alpha=0.12, lw=0)
    b1.plot(xN, np.array([r["I_iolab"] for r in newton]) * 1e4, "o", color=BLUE, ms=4, label="Method N")
    b1.plot(xE, np.array([r["I_iolab"] for r in energy]) * 1e4, "s", color=ORANGE, ms=4, label="Method E")
    b1.axhline(I_box * 1e4, color=MUTED, ls=":", lw=1)
    b1.axhline(I_shell * 1e4, color=MUTED, ls=":", lw=1)
    b1.text(0.55, I_box * 1e4, "box, uniform", va="bottom", fontsize=7, color=MUTED)
    b1.text(0.55, I_shell * 1e4, "box, mass in walls", va="bottom", fontsize=7, color=MUTED)
    b1.set_xlabel("run")
    b1.set_xticks(range(1, max(len(newton), len(energy)) + 1))
    b1.set_ylabel(r"$I_{iOLab}$ ($10^{-4}$ kg m$^2$)")
    names = {"s": r"$\sin\theta$", "k": r"$k$", "I_rolls": r"$I_{rolls}$", "model": "roll model", "R": r"$R$",
             "a_meas": r"$\bar\alpha$ (runs)", "w_meas": r"$\omega_B$ (runs)", "D_gate": r"$s_{AB}$",
             "eps": r"$\varepsilon$", "fdrag": "air drag", "drag": "air drag", "M": r"$M$"}
    order = ["s", "k", "I_rolls", "model", "R", "a_meas", "w_meas", "D_gate", "eps", "fdrag", "M"]
    y = np.arange(len(order))
    tN = [abs(termsN.get(k_, termsN.get("drag", 0.0) if k_ == "fdrag" else 0.0)) * 1e5 for k_ in order]
    tE = [abs(termsE.get(k_, 0.0)) * 1e5 for k_ in order]
    b2.barh(y - 0.18, tN, height=0.36, color=BLUE, label="Method N")
    b2.barh(y + 0.18, tE, height=0.36, color=ORANGE, label="Method E")
    b2.set_yticks(y, [names[k_] for k_ in order])
    b2.invert_yaxis()
    b2.set_xlabel(r"term in $\delta I_{iOLab}$ ($10^{-5}$ kg m$^2$)")
    fig.legend(*b1.get_legend_handles_labels(), loc="upper center", ncol=2, fontsize=8, frameon=False,
               bbox_to_anchor=(0.5, 1.04))
    fig.savefig(OUT_DIR / "report_figure_results.png", dpi=300, bbox_inches="tight")
    fig.savefig(OUT_DIR / "report_figure_results.pdf", bbox_inches="tight")
    plt.show()

## What to take into the report

From the `output` folder:

1. `parts_table.csv`, `calibration_table.csv`, `angle_table.csv`: the measured inputs. Copy them into one table with units and uncertainties.
2. `newton_table.csv` and `energy_table.csv`: one row per run. Copy the columns you need into the run tables.
3. `report_figure_runs.png`: one Newton run with its two windows, and one energy run with the corner at the board end. Write your own caption.
4. `report_figure_results.png`: $I_{iOLab}$ for every run of both methods, and the uncertainty budget.
5. `results_summary.txt`: the final numbers, the agreement test and the box model range.

Round every number with its uncertainty, keep the units, and say in the text which run the figures show.